# Step 2: Model Training
Train a RandomForest classifier on the cleaned loan data.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib
import boto3
import os
from io import StringIO, BytesIO

S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "http://minio:9000")
S3_BUCKET = os.environ.get("S3_BUCKET", "pipelines")
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "minio")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "minio123")

s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT,
                   aws_access_key_id=AWS_ACCESS_KEY_ID,
                   aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
                   verify=False)

INPUT_KEY = os.environ.get("INPUT_KEY", "pipeline-artifacts/cleaned-data.csv")
MODEL_KEY = os.environ.get("MODEL_KEY", "pipeline-artifacts/model.joblib")
TEST_DATA_KEY = os.environ.get("TEST_DATA_KEY", "pipeline-artifacts/test-data.csv")

obj = s3.get_object(Bucket=S3_BUCKET, Key=INPUT_KEY)
df = pd.read_csv(obj["Body"])
print(f"Training data: {len(df)} rows from s3://{S3_BUCKET}/{INPUT_KEY}")

In [ ]:
target = "approved"
X = df.drop(columns=[target, "loan_id"], errors="ignore")
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
clf.fit(X_train, y_train)

print(f"Trained on {len(X_train)} samples")
print(f"Training accuracy: {clf.score(X_train, y_train):.4f}")

In [ ]:
model_buf = BytesIO()
joblib.dump(clf, model_buf)
model_buf.seek(0)
s3.put_object(Bucket=S3_BUCKET, Key=MODEL_KEY, Body=model_buf.getvalue())
print(f"Model saved to s3://{S3_BUCKET}/{MODEL_KEY}")

test_df = X_test.copy()
test_df[target] = y_test
csv_buf = StringIO()
test_df.to_csv(csv_buf, index=False)
s3.put_object(Bucket=S3_BUCKET, Key=TEST_DATA_KEY, Body=csv_buf.getvalue())
print(f"Test data saved to s3://{S3_BUCKET}/{TEST_DATA_KEY}")